[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_03_engineering_workflow/ch15_git_clean_code/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 15 Walkthrough: Git, Clean Code, Type Hints & Research Templates

**Book Reference:** Chapter 15 (Part III Capstone), Sections 15.1 – 15.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every file and logging example runs cleanly inside `tempfile.TemporaryDirectory()` with explicitly closed handlers so zero stray files remain.

## Section 15.3.1 — `.gitignore` Coverage Auditing & Secret Scanning

In [ ]:
import fnmatch

gitignore_text = (
    "# Never commit raw clinical data or secrets\n"
    "*.csv\n"
    ".env\n"
    "*.log\n"
    "__pycache__/*\n"
)
staged_candidates = [
    "README.md",
    "src/clean_cohort.py",
    "raw_mch_2026.csv",
    ".env",
    "run.log",
]

patterns = [
    line.strip()
    for line in gitignore_text.splitlines()
    if line.strip() and not line.strip().startswith("#")
]
tracked = sorted(
    fname for fname in staged_candidates
    if not any(fnmatch.fnmatch(fname, pat) for pat in patterns)
)
print("Active .gitignore patterns:", patterns)
print("Safe to track in Git:", tracked)

config_sample = (
    "STUDY_ID = 'MCH-BD-2026'\n"
    "TOKEN = 'ghp_exampleLeak999'\n"
    "API_KEY=demo_secret_token\n"
)
secret_tags = ("ghp_", "AIza", "sk-", "API_KEY=", "PASSWORD=")
flagged_lines = [
    idx for idx, ln in enumerate(config_sample.splitlines(), start=1)
    if any(tag in ln for tag in secret_tags)
]
print("Flagged secret lines (1-based):", flagged_lines)


## Section 15.3.2 — Clean Code, Type Hints, and `TypedDict` Contracts

In [ ]:
from typing import Optional, TypedDict


class CohortSummary(TypedDict):
    site: str
    valid_n: int
    mean_hb: Optional[float]


def summarize_site_hb(
    site: str,
    readings: list[float],
) -> CohortSummary:
    """Return a typed hemoglobin summary for one clinic site."""
    clean = [val for val in readings if 3.0 <= val <= 22.0]
    mean_val: Optional[float] = (
        round(sum(clean) / len(clean), 2) if clean else None
    )
    return {
        "site": site.strip().title(),
        "valid_n": len(clean),
        "mean_hb": mean_val,
    }


dhaka_report = summarize_site_hb("  dhaka ", [10.4, 999.0, 11.6])
empty_report = summarize_site_hb("sylhet", [999.0])
print("Dhaka summary:", dhaka_report)
print("Empty summary:", empty_report)


## Section 15.3.3 — Structured `logging` vs. `print()` (With Clean Handler Teardown)

In [ ]:
import logging
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    log_path = Path(tmp) / "cohort_run.log"
    logger = logging.getLogger("ch15_walkthrough_demo")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fmt = logging.Formatter("[%(levelname)s] %(message)s")
    handler.setFormatter(fmt)
    logger.addHandler(handler)

    try:
        logger.info("Pipeline started for Dhaka ANC cohort")
        logger.warning("Row 4: skipped sentinel hb_g_dl='999'")
        logger.info("Pipeline completed: 2 valid, 1 skipped")
    finally:
        logger.removeHandler(handler)
        handler.close()

    print(log_path.read_text(encoding="utf-8").strip())


## Section 15.3.4 — Cryptographic Dataset Provenance (`hashlib.sha256`)

In [ ]:
import hashlib
from pathlib import Path
import tempfile


def sha256_file(filepath: Path) -> str:
    """Compute lowercase 64-char SHA-256 hex digest in 64 KB chunks."""
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()


with tempfile.TemporaryDirectory() as tmp:
    raw_csv = Path(tmp) / "mch_v1.csv"
    raw_csv.write_text("id,hb_g_dl\nP1,10.8\n", encoding="utf-8")
    digest_v1 = sha256_file(raw_csv)

    # Flip a single digit (10.8 -> 10.9) to demonstrate avalanche effect
    raw_csv.write_text("id,hb_g_dl\nP1,10.9\n", encoding="utf-8")
    digest_v2 = sha256_file(raw_csv)

    print("Digest v1 (first 16):", digest_v1[:16], "len=", len(digest_v1))
    print("Digest v2 (first 16):", digest_v2[:16], "len=", len(digest_v2))
    print("Identical files?:", digest_v1 == digest_v2)


## Section 15.3.5 — Break It On Purpose! (Provenance & Logging Error Anatomy)

In [ ]:
# Intentional Bug #1: FileNotFoundError — hashing a missing dataset
import hashlib
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    missing_csv = Path(tmp) / "missing_cohort.csv"
    with open(missing_csv, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()


In [ ]:
# Intentional Bug #2: TypeError — passing str instead of bytes to sha256
import hashlib

csv_text = "patient_id,hb_g_dl\nP01,11.2\n"
digest = hashlib.sha256(csv_text).hexdigest()


In [ ]:
# Intentional Bug #3: ValueError — invalid logging level name
import logging

bad_logger = logging.getLogger("ch15_bad_level_demo")
bad_logger.setLevel("URGENT")


## Section 15.4 — Worked Blueprint: Pre-Submission Research Release Gatekeeper

In [ ]:
import hashlib
import json
import logging
from pathlib import Path
import tempfile
from typing import TypedDict


class ReleaseAudit(TypedDict):
    data_files: int
    manifest_path: str
    secret_leaks: int
    release_ready: bool


def prepare_release_bundle(
    repo_dir: Path,
    log_path: Path,
) -> ReleaseAudit:
    """Build a SHA-256 manifest, scan src/ for secrets, and log audit."""
    logger = logging.getLogger("ch15_release_gate")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)

    secret_tags = ("ghp_", "AIza", "sk-", "API_KEY=", "PASSWORD=")
    manifest: dict[str, dict[str, object]] = {}
    leaks = 0

    try:
        data_dir = repo_dir / "data"
        for p in sorted(data_dir.iterdir(), key=lambda x: x.name):
            if p.is_file() and p.suffix.lower() in {".csv", ".json"}:
                digest = hashlib.sha256(p.read_bytes()).hexdigest()
                manifest[p.name] = {
                    "bytes": p.stat().st_size,
                    "sha256": digest,
                }
        logger.info("Hashed %d dataset file(s)", len(manifest))

        src_dir = repo_dir / "src"
        for py_file in sorted(src_dir.glob("*.py")):
            text = py_file.read_text(encoding="utf-8")
            for ln_no, line in enumerate(text.splitlines(), start=1):
                if any(tag in line for tag in secret_tags):
                    leaks += 1
                    logger.warning(
                        "Leak in %s line %d", py_file.name, ln_no
                    )

        manifest_file = repo_dir / "manifest.json"
        with open(manifest_file, "w", encoding="utf-8") as out_f:
            json.dump(manifest, out_f, indent=2)

        ready = len(manifest) > 0 and leaks == 0
        logger.info("Release ready: %s", ready)
    finally:
        logger.removeHandler(handler)
        handler.close()

    return {
        "data_files": len(manifest),
        "manifest_path": manifest_file.name,
        "secret_leaks": leaks,
        "release_ready": ready,
    }


with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    (repo / "data").mkdir()
    (repo / "src").mkdir()
    (repo / "data" / "mch_clean.csv").write_text(
        "id,hb_g_dl\nP1,11.4\n", encoding="utf-8"
    )
    (repo / "src" / "analyze.py").write_text(
        "STUDY = 'MCH-BD-2026'\n", encoding="utf-8"
    )
    audit_log = repo / "release_audit.log"
    result = prepare_release_bundle(repo, audit_log)
    print("Audit Result:\n" + json.dumps(result, indent=2))
    print("Audit Log:\n" + audit_log.read_text(encoding="utf-8").strip())
